# Triagegeist: Safety-Focused Acuity Support for Emergency Triage

## Executive Summary

This notebook builds a reproducible emergency department triage support prototype for the Triagegeist hackathon. The goal is not only to predict an ESI-style acuity label (`1` = most urgent, `5` = least urgent), but to evaluate the model as a safety layer against dangerous under-triage.

**Clinical framing:** a wrong prediction is most dangerous when a true high-acuity patient (`acuity <= 2`) is assigned to a lower-priority class (`3-5`). The notebook therefore reports standard multiclass metrics and a dedicated severe under-triage metric.

**Kaggle-verified run:** the full notebook completed in `3m 27s` in Kaggle and produced all expected output artifacts.

**Key results from the Kaggle run:**

| Result | Value |
|---|---:|
| Holdout accuracy, full valid model | 0.9946 |
| Holdout macro-F1, full valid model | 0.9840 |
| Holdout quadratic weighted kappa | 0.9975 |
| Holdout severe under-triage | 0.0000 |
| 5-fold CV mean macro-F1 | 0.9564 |
| 5-fold CV mean severe under-triage | 0.00024 |
| High-risk alert recall at threshold 0.50 | 1.0000 |
| Missed high-acuity cases at threshold 0.50 | 0 |

The strongest practical output is a **high-risk alert mode** using `P(acuity <= 2)`: on the fixed validation split, a threshold of `0.50` captured every true acuity 1-2 patient with one false alert among 16,000 validation records.

## Setup

The notebook runs locally from `triagegeist-data/` and on Kaggle if the competition dataset is attached under `/kaggle/input/...`.

## How This Notebook Maps To The Judging Rubric

- **Clinical relevance:** focuses on under-triage, the safety-critical failure mode in emergency triage.
- **Technical quality:** uses one shared preprocessing path for validation and submission, excludes train-only outcomes, and reports holdout plus cross-validation metrics.
- **Documentation quality:** each section states why the step matters clinically and technically.
- **Insight and findings:** includes ablations, high-risk threshold analysis, subgroup safety checks, and error review.
- **Novelty and impact:** reframes multiclass acuity prediction as an auditable high-risk alert layer for clinician review.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

In [ ]:
def find_data_dir():
    candidates = [
        Path('triagegeist-data'),
        Path('/kaggle/input/triagegeist'),
        Path('/kaggle/input/triagegeist-data'),
    ]
    for candidate in candidates:
        if (candidate / 'train.csv').exists():
            return candidate

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob('train.csv'))
        if matches:
            return matches[0].parent

    raise FileNotFoundError('Could not find train.csv. Attach the Triagegeist dataset or run from the project root.')

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR

## Load Data And Guard Against Leakage

All files join on `patient_id`. The key modeling constraint is that `disposition` and `ed_los_hours` appear only in train and are post-triage outcomes. They are useful for exploratory context, but not valid test-time features. This notebook excludes them from every model used for validation or submission.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
chief_complaints = pd.read_csv(DATA_DIR / 'chief_complaints.csv')
patient_history = pd.read_csv(DATA_DIR / 'patient_history.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

shapes = pd.DataFrame({
    'table': ['train', 'test', 'chief_complaints', 'patient_history', 'sample_submission'],
    'rows': [len(train), len(test), len(chief_complaints), len(patient_history), len(sample_submission)],
    'columns': [train.shape[1], test.shape[1], chief_complaints.shape[1], patient_history.shape[1], sample_submission.shape[1]],
})
shapes

In [ ]:
checks = {
    'train_id_unique': train['patient_id'].is_unique,
    'test_id_unique': test['patient_id'].is_unique,
    'chief_complaints_id_unique': chief_complaints['patient_id'].is_unique,
    'patient_history_id_unique': patient_history['patient_id'].is_unique,
    'train_test_overlap': len(set(train['patient_id']) & set(test['patient_id'])),
    'missing_complaints_train': (~train['patient_id'].isin(chief_complaints['patient_id'])).sum(),
    'missing_complaints_test': (~test['patient_id'].isin(chief_complaints['patient_id'])).sum(),
    'missing_history_train': (~train['patient_id'].isin(patient_history['patient_id'])).sum(),
    'missing_history_test': (~test['patient_id'].isin(patient_history['patient_id'])).sum(),
    'train_only_columns': sorted(set(train.columns) - set(test.columns)),
}
checks

## Target, Class Balance, And Missingness

`triage_acuity` is ordinal and clinically asymmetric: confusing acuity 1 with 2 is less dangerous than sending a true acuity 1-2 patient into classes 3-5. Missingness is also clinically meaningful because lower-acuity patients may receive less complete measurement at intake.

In [ ]:
target_distribution = pd.DataFrame({
    'count': train['triage_acuity'].value_counts().sort_index(),
    'rate': train['triage_acuity'].value_counts(normalize=True).sort_index(),
})
target_distribution

In [ ]:
missing_cols = [
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure',
    'shock_index', 'respiratory_rate', 'temperature_c'
]
missing_by_acuity = train.groupby('triage_acuity')[missing_cols].apply(lambda frame: frame.isna().mean()).round(3)
missing_by_acuity

In [ ]:
clinical_gradient_cols = [
    'age', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c',
    'spo2', 'gcs_total', 'pain_score', 'shock_index', 'news2_score', 'num_comorbidities'
]
train.groupby('triage_acuity')[clinical_gradient_cols].mean().round(2)

## Feature Engineering

The preprocessing is shared between validation and final submission. Train-only outcomes are excluded before modeling.

In [ ]:
VITAL_MISSING_COLS = [
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure',
    'shock_index', 'respiratory_rate', 'temperature_c'
]
OUTCOME_COLS = ['triage_acuity', 'disposition', 'ed_los_hours']
COMPOSITE_COLS = ['mean_arterial_pressure', 'pulse_pressure', 'shock_index', 'news2_score', 'bmi', 'age_group']


def build_frame(df):
    out = (
        df.merge(chief_complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')
          .merge(patient_history, on='patient_id', how='left')
          .copy()
    )
    out['chief_complaint_raw'] = out['chief_complaint_raw'].fillna('')
    out['pain_score_missing'] = out['pain_score'].eq(-1).astype(int)
    out['pain_score'] = out['pain_score'].mask(out['pain_score'].eq(-1), np.nan)

    for col in VITAL_MISSING_COLS:
        out[f'{col}_missing'] = out[col].isna().astype(int)

    out['hypotension_flag'] = out['systolic_bp'].lt(90).fillna(False).astype(int)
    out['hypoxia_flag'] = out['spo2'].lt(92).fillna(False).astype(int)
    out['tachycardia_flag'] = out['heart_rate'].gt(110).fillna(False).astype(int)
    out['fever_flag'] = out['temperature_c'].ge(38).fillna(False).astype(int)
    out['altered_mental_status_flag'] = out['mental_status_triage'].ne('alert').astype(int)
    return out


def feature_columns(frame, variant='all_valid_features'):
    cols = [c for c in frame.columns if c not in OUTCOME_COLS + ['patient_id']]

    if variant == 'main_table_only':
        return [c for c in cols if not c.startswith('hx_') and c != 'chief_complaint_raw']

    if variant == 'main_plus_history_no_text':
        return [c for c in cols if c != 'chief_complaint_raw']

    if variant == 'no_composite_scores':
        return [c for c in cols if c not in COMPOSITE_COLS]

    if variant == 'minimal_clinical_intake':
        keep = [
            'arrival_mode', 'arrival_hour', 'age', 'sex', 'systolic_bp', 'diastolic_bp',
            'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total',
            'pain_score', 'mental_status_triage', 'chief_complaint_raw',
            'num_prior_ed_visits_12m', 'num_prior_admissions_12m',
            'num_active_medications', 'num_comorbidities', 'pain_score_missing',
            'hypotension_flag', 'hypoxia_flag', 'tachycardia_flag', 'fever_flag',
            'altered_mental_status_flag',
        ]
        keep += [c for c in frame.columns if c.startswith('hx_')]
        return [c for c in keep if c in frame.columns]

    if variant == 'all_valid_features':
        return cols

    raise ValueError(f'Unknown variant: {variant}')

train_model = build_frame(train)
test_model = build_frame(test)
feature_columns(train_model)[:10], len(feature_columns(train_model))

In [ ]:
def build_pipeline(X):
    categorical_cols = [c for c in X.select_dtypes(include='object').columns if c != 'chief_complaint_raw']
    has_text = 'chief_complaint_raw' in X.columns
    numeric_cols = [c for c in X.columns if c not in categorical_cols + (['chief_complaint_raw'] if has_text else [])]

    transformers = []
    if numeric_cols:
        transformers.append((
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                ('scaler', StandardScaler(with_mean=False)),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10)),
            ]),
            categorical_cols,
        ))
    if has_text:
        transformers.append((
            'text',
            TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=3000, sublinear_tf=True),
            'chief_complaint_raw',
        ))

    return Pipeline([
        ('preprocess', ColumnTransformer(transformers=transformers, sparse_threshold=0.3)),
        ('model', SGDClassifier(
            loss='log_loss',
            alpha=1e-5,
            penalty='l2',
            class_weight='balanced',
            max_iter=2000,
            tol=1e-4,
            random_state=SEED,
            n_jobs=-1,
        )),
    ])


def metric_row(name, y_true, y_pred, raw_feature_count):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    high_acuity = y_true <= 2
    return {
        'variant': name,
        'raw_feature_count': raw_feature_count,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'mae_ordinal': mean_absolute_error(y_true, y_pred),
        'quadratic_weighted_kappa': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        'undertriage_rate_all': (y_pred > y_true).mean(),
        'severe_undertriage_true_1_2_to_3_5': (y_pred[high_acuity] >= 3).mean(),
    }

## Validation And Ablations

The ablation table answers the main methodological question: what information actually drives safety? Structured intake data is useful, but chief complaint text sharply improves high-acuity recognition. This matters for the clinical story because triage notes often contain the first signal of shock, overdose, meningitis, anaphylaxis, or other urgent presentations.

In [ ]:
y = train_model['triage_acuity']
idx_train, idx_val = train_test_split(train_model.index, test_size=0.2, random_state=SEED, stratify=y)

rows = []
majority_class = int(y.loc[idx_train].mode()[0])
rows.append(metric_row('majority_class_3', y.loc[idx_val], np.full(len(idx_val), majority_class), 0))

fitted_models = {}
for variant in [
    'main_table_only',
    'main_plus_history_no_text',
    'all_valid_features',
    'no_composite_scores',
    'minimal_clinical_intake',
]:
    cols = feature_columns(train_model, variant)
    X_train = train_model.loc[idx_train, cols]
    X_val = train_model.loc[idx_val, cols]

    pipe = build_pipeline(X_train)
    pipe.fit(X_train, y.loc[idx_train])
    pred = pipe.predict(X_val)

    fitted_models[variant] = (pipe, cols, pred)
    rows.append(metric_row(variant, y.loc[idx_val], pred, len(cols)))

ablation_metrics = pd.DataFrame(rows)
ablation_metrics.to_csv(OUTPUT_DIR / 'ablation_metrics.csv', index=False)
ablation_metrics.round(4)

## Cross-Validation Robustness

A single split can overstate stability, so the full valid-feature model is also evaluated with 5-fold stratified CV. One fold is weaker than the others, which is exactly why the notebook reports variance instead of overclaiming a single score. The safety metric remains very low across folds.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_cols = feature_columns(train_model, 'all_valid_features')
X_cv = train_model[cv_cols]
y_cv = train_model['triage_acuity']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for fold, (train_pos, val_pos) in enumerate(cv.split(X_cv, y_cv), start=1):
    pipe = build_pipeline(X_cv.iloc[train_pos])
    pipe.fit(X_cv.iloc[train_pos], y_cv.iloc[train_pos])
    fold_pred = pipe.predict(X_cv.iloc[val_pos])
    row = metric_row(f'fold_{fold}', y_cv.iloc[val_pos], fold_pred, len(cv_cols))
    row['fold'] = fold
    row['train_rows'] = len(train_pos)
    row['validation_rows'] = len(val_pos)
    cv_rows.append(row)

cv_metrics = pd.DataFrame(cv_rows)
metric_cols = [
    'accuracy', 'macro_f1', 'weighted_f1', 'mae_ordinal',
    'quadratic_weighted_kappa', 'undertriage_rate_all',
    'severe_undertriage_true_1_2_to_3_5',
]
cv_summary = cv_metrics[metric_cols].agg(['mean', 'std']).T.reset_index().rename(columns={'index': 'metric'})
cv_metrics.to_csv(OUTPUT_DIR / 'cv_metrics.csv', index=False)
cv_summary.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
cv_summary.round(5)

In [ ]:
best_pipe, best_cols, val_pred = fitted_models['all_valid_features']
labels = [1, 2, 3, 4, 5]
confusion = pd.DataFrame(
    confusion_matrix(y.loc[idx_val], val_pred, labels=labels),
    index=[f'true_{x}' for x in labels],
    columns=[f'pred_{x}' for x in labels],
)
print(classification_report(y.loc[idx_val], val_pred, labels=labels, digits=4))
confusion

## High-Risk Alert Operating Points

For clinical use, the most plausible workflow is not automatic replacement of the nurse's score. It is a high-risk alert: if `P(acuity <= 2)` crosses a threshold, the patient is surfaced for immediate clinician review. The threshold table shows the tradeoff between recall, precision, and alert burden.

In [ ]:
y_val = y.loc[idx_val]
val_proba = best_pipe.predict_proba(train_model.loc[idx_val, best_cols])
class_labels = best_pipe.named_steps['model'].classes_
high_risk_score = val_proba[:, np.isin(class_labels, [1, 2])].sum(axis=1)
y_high = y_val.le(2).to_numpy()


def high_risk_metric_row(threshold):
    alert = high_risk_score >= threshold
    tp = int((alert & y_high).sum())
    fp = int((alert & ~y_high).sum())
    fn = int((~alert & y_high).sum())
    tn = int((~alert & ~y_high).sum())
    return {
        'threshold': float(threshold),
        'alert_rate': float(alert.mean()),
        'recall_high_acuity': tp / (tp + fn) if (tp + fn) else np.nan,
        'precision_alert': tp / (tp + fp) if (tp + fp) else np.nan,
        'specificity_non_high_acuity': tn / (tn + fp) if (tn + fp) else np.nan,
        'missed_high_acuity_count': fn,
        'alerted_high_acuity_count': tp,
        'false_alert_count': fp,
    }

threshold_grid = np.round(np.linspace(0.05, 0.95, 19), 2)
threshold_metrics = pd.DataFrame([high_risk_metric_row(t) for t in threshold_grid])
threshold_metrics.to_csv(OUTPUT_DIR / 'high_risk_threshold_grid.csv', index=False)

candidate_thresholds = np.unique(np.r_[0, high_risk_score, 1])
dense_threshold_metrics = pd.DataFrame([high_risk_metric_row(t) for t in candidate_thresholds])
operating_points = []
for target_recall in [0.95, 0.98, 0.99, 1.00]:
    candidates = dense_threshold_metrics[dense_threshold_metrics['recall_high_acuity'] >= target_recall]
    if not candidates.empty:
        selected = candidates.sort_values(['threshold'], ascending=False).iloc[0].copy()
        selected['target_recall'] = target_recall
        operating_points.append(selected)

operating_points = pd.DataFrame(operating_points)
operating_points.to_csv(OUTPUT_DIR / 'high_risk_operating_points.csv', index=False)
operating_points[[
    'target_recall', 'threshold', 'alert_rate', 'recall_high_acuity',
    'precision_alert', 'specificity_non_high_acuity', 'missed_high_acuity_count'
]].round(4)

## Error Review

The fixed validation split has no severe under-triage errors. The remaining mistakes are mostly boundary errors within the high-risk zone, especially `1 <-> 2`. Those errors still deserve audit, but they are less dangerous than classifying a true acuity 1-2 patient as 3-5.

In [ ]:
review_cols = [
    'patient_id', 'chief_complaint_raw', 'triage_acuity', 'age', 'sex', 'arrival_mode',
    'mental_status_triage', 'systolic_bp', 'diastolic_bp', 'heart_rate',
    'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score',
    'shock_index', 'news2_score', 'num_comorbidities',
]
error_review = train_model.loc[idx_val, review_cols].copy()
error_review['predicted_acuity'] = val_pred
error_review['high_risk_score'] = high_risk_score
error_review['acuity_error'] = error_review['predicted_acuity'] - error_review['triage_acuity']
error_review['error_type'] = np.select(
    [
        error_review['acuity_error'].gt(0),
        error_review['acuity_error'].lt(0),
    ],
    ['undertriage', 'overtriage'],
    default='correct',
)
error_review['severe_undertriage'] = (
    error_review['triage_acuity'].le(2) & error_review['predicted_acuity'].ge(3)
)

errors_only = (
    error_review[error_review['acuity_error'].ne(0)]
    .sort_values(['severe_undertriage', 'triage_acuity', 'acuity_error'], ascending=[False, True, False])
    .reset_index(drop=True)
)
error_summary = (
    error_review.groupby(['triage_acuity', 'predicted_acuity'])
    .size()
    .reset_index(name='count')
    .sort_values(['triage_acuity', 'predicted_acuity'])
)

errors_only.to_csv(OUTPUT_DIR / 'error_review.csv', index=False)
error_summary.to_csv(OUTPUT_DIR / 'error_summary.csv', index=False)

print(f'Total validation errors: {len(errors_only)}')
print(f'Severe under-triage errors: {int(errors_only["severe_undertriage"].sum())}')
errors_only[[
    'patient_id', 'triage_acuity', 'predicted_acuity', 'acuity_error', 'error_type',
    'high_risk_score', 'chief_complaint_raw', 'mental_status_triage',
    'systolic_bp', 'heart_rate', 'spo2', 'gcs_total', 'news2_score'
]].head(20)

## Safety Subgroup Check

A triage support system should be audited for uneven safety across patient groups and workflow contexts. The table below checks whether severe under-triage concentrates by age group, sex, language, insurance type, site, or arrival mode.

In [ ]:
val_eval = train_model.loc[idx_val, ['triage_acuity', 'age_group', 'sex', 'language', 'insurance_type', 'site_id', 'arrival_mode']].copy()
val_eval['predicted_acuity'] = val_pred
val_eval['severe_undertriage'] = (val_eval['triage_acuity'].le(2) & val_eval['predicted_acuity'].ge(3)).astype(int)
val_eval['high_acuity'] = val_eval['triage_acuity'].le(2).astype(int)

subgroup_rows = []
for col in ['age_group', 'sex', 'language', 'insurance_type', 'site_id', 'arrival_mode']:
    grouped = val_eval.groupby(col).agg(
        rows=('triage_acuity', 'size'),
        high_acuity_cases=('high_acuity', 'sum'),
        severe_undertriage_cases=('severe_undertriage', 'sum'),
    )
    grouped['severe_undertriage_rate_among_high_acuity'] = np.where(
        grouped['high_acuity_cases'] > 0,
        grouped['severe_undertriage_cases'] / grouped['high_acuity_cases'],
        np.nan,
    )
    grouped.insert(0, 'group', col)
    subgroup_rows.append(grouped.reset_index().rename(columns={col: 'value'}))

subgroup_safety = pd.concat(subgroup_rows, ignore_index=True)
subgroup_safety.to_csv(OUTPUT_DIR / 'subgroup_safety.csv', index=False)
subgroup_safety.sort_values(['group', 'severe_undertriage_rate_among_high_acuity', 'high_acuity_cases'], ascending=[True, False, False]).head(30)

## Interpretable Signals

The sparse linear model allows a quick audit of learned signals. These coefficients are not causal explanations, but they help verify that high-acuity predictions are driven by clinically plausible features and complaint terms rather than train-only outcomes.

In [ ]:
feature_names = best_pipe.named_steps['preprocess'].get_feature_names_out()
classes = best_pipe.named_steps['model'].classes_
coefs = best_pipe.named_steps['model'].coef_

top_features = []
for class_idx, cls in enumerate(classes):
    weights = pd.Series(coefs[class_idx], index=feature_names).sort_values(ascending=False)
    for rank, (feature, weight) in enumerate(weights.head(12).items(), start=1):
        top_features.append({'acuity_class': int(cls), 'rank': rank, 'feature': feature, 'weight': weight})

top_features = pd.DataFrame(top_features)
top_features.to_csv(OUTPUT_DIR / 'top_linear_features.csv', index=False)
top_features.head(24)

## Visual Artifacts

These figures are saved for the writeup and project repository. They are intentionally simple: target balance, ablation results, high-risk threshold tradeoff, confusion matrix, and a cover image.

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / '.matplotlib'))
(OUTPUT_DIR / '.matplotlib').mkdir(exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 180,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

# Target distribution.
fig, ax = plt.subplots(figsize=(6.5, 3.6))
rates = target_distribution['rate'] * 100
ax.bar(rates.index.astype(str), rates.values, color=['#b63b3b', '#d47842', '#e5b84b', '#5e9f6e', '#4f83b5'])
ax.set_title('Triage acuity distribution')
ax.set_xlabel('ESI-style acuity: 1 = most urgent, 5 = least urgent')
ax.set_ylabel('Training records (%)')
for x, v in enumerate(rates.values):
    ax.text(x, v + 0.6, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'target_distribution.png', bbox_inches='tight')
plt.close(fig)

# Ablation metrics.
plot_ablation = ablation_metrics.copy()
plot_ablation['label'] = plot_ablation['variant'].str.replace('_', ' ', regex=False)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
y_pos = np.arange(len(plot_ablation))
ax.barh(y_pos - 0.18, plot_ablation['macro_f1'], height=0.34, label='Macro-F1', color='#4f83b5')
ax.barh(y_pos + 0.18, 1 - plot_ablation['severe_undertriage_true_1_2_to_3_5'], height=0.34, label='1 - severe under-triage rate', color='#5e9f6e')
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_ablation['label'])
ax.set_xlim(0, 1.03)
ax.set_title('Ablation results: model quality and high-acuity safety')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'ablation_metrics.png', bbox_inches='tight')
plt.close(fig)

# High-risk threshold tradeoff.
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(threshold_metrics['threshold'], threshold_metrics['recall_high_acuity'], marker='o', label='Recall: true acuity 1-2', color='#b63b3b')
ax.plot(threshold_metrics['threshold'], threshold_metrics['precision_alert'], marker='o', label='Alert precision', color='#4f83b5')
ax.plot(threshold_metrics['threshold'], threshold_metrics['alert_rate'], marker='o', label='Alert rate', color='#d47842')
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('High-risk alert threshold')
ax.set_ylabel('Rate')
ax.set_title('High-risk alert threshold tradeoff')
ax.legend(loc='best')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'high_risk_threshold_tradeoff.png', bbox_inches='tight')
plt.close(fig)

# Confusion matrix.
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(confusion.values, cmap='Blues')
ax.set_title('Validation confusion matrix')
ax.set_xlabel('Predicted acuity')
ax.set_ylabel('True acuity')
ax.set_xticks(np.arange(5), labels=[1, 2, 3, 4, 5])
ax.set_yticks(np.arange(5), labels=[1, 2, 3, 4, 5])
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        value = confusion.values[i, j]
        color = 'white' if value > confusion.values.max() * 0.45 else '#1f2933'
        ax.text(j, i, f'{value:,}', ha='center', va='center', fontsize=9, color=color)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.close(fig)

# Cover image, 560 x 280 px at 140 dpi = 4 x 2 inches.
fig, ax = plt.subplots(figsize=(4, 2))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.add_patch(plt.Rectangle((0, 0), 1, 1, color='#f6f3ee'))
ax.add_patch(plt.Rectangle((0, 0.78), 1, 0.22, color='#254f5f'))
ax.text(0.05, 0.88, 'TRIAGEGEIST', color='white', fontsize=15, weight='bold', va='center')
ax.text(0.05, 0.68, 'Safety-focused acuity support', color='#1f2933', fontsize=12, weight='bold')
ax.text(0.05, 0.54, 'Predict ESI 1-5 and measure dangerous under-triage', color='#374151', fontsize=8.4)
for x, h, color in [(0.08, 0.22, '#b63b3b'), (0.18, 0.32, '#d47842'), (0.28, 0.48, '#e5b84b'), (0.38, 0.38, '#5e9f6e'), (0.48, 0.28, '#4f83b5')]:
    ax.add_patch(plt.Rectangle((x, 0.14), 0.055, h, color=color))
ax.plot([0.62, 0.68, 0.72, 0.78, 0.84, 0.90, 0.96], [0.30, 0.30, 0.48, 0.18, 0.58, 0.30, 0.30], color='#b63b3b', linewidth=2.0)
ax.text(0.62, 0.12, 'high-risk alert: acuity <= 2', color='#374151', fontsize=7.8)
fig.savefig(FIG_DIR / 'cover_image_560x280.png', dpi=140)
plt.close(fig)

sorted(str(p) for p in FIG_DIR.glob('*.png'))

## Final Training And Submission

After validation and audit, the final model is refit on all training rows and used to create `submission.csv`. Even though this is a judged hackathon rather than a leaderboard competition, the submission file is still a reproducibility artifact showing that the pipeline runs end-to-end on the test set.

In [ ]:
final_cols = feature_columns(train_model, 'all_valid_features')
final_pipe = build_pipeline(train_model[final_cols])
final_pipe.fit(train_model[final_cols], train_model['triage_acuity'])

test_predictions = final_pipe.predict(test_model[final_cols]).astype(int)
submission = pd.DataFrame({
    'patient_id': test_model['patient_id'],
    'triage_acuity': test_predictions,
})

assert submission.shape == sample_submission.shape
assert submission['patient_id'].equals(sample_submission['patient_id'])
assert set(submission['triage_acuity']).issubset({1, 2, 3, 4, 5})

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
submission['triage_acuity'].value_counts().sort_index(), submission_path

## Final Takeaways For The Writeup

The strongest version of this project is a safety-focused triage support study:

- The model is trained only on triage-time information; train-only outcomes are excluded.
- Chief complaint text is the dominant signal in this synthetic dataset, and ablations make that explicit.
- Severe under-triage is treated as the primary clinical safety metric.
- The high-risk alert mode is the clearest impact path: surface likely acuity 1-2 patients for immediate review.
- Cross-validation variance is reported honestly, and the dataset is clearly described as synthetic.
- This is a reproducible research prototype, not a deployable medical device.